In [1]:
!pip install -q chromadb sentence-transformers groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
import os
import chromadb
from groq import Groq
from google.colab import userdata

# Setup

In [3]:
chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name='ABC_Handbook')
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [6]:
# Document: ABC Employee Handbook (chunked by section)
handbook_chunks = [
    "Company Overview: ABC Corp was founded in 2010 with a mission to innovate in the technology sector. We prioritize transparency, collaboration, and continuous learning for all employees.",
    "Remote Work Policy: Employees are eligible for remote work up to 3 days per week. Requests must be approved by the direct manager, and employees are expected to be available during core business hours of 10:00 AM to 3:00 PM EST.",
    "Leave Policy: All full-time employees are entitled to 20 days of Paid Time Off (PTO) per year, accrued monthly. Sick leave is unlimited but requires a doctor's note for absences exceeding three consecutive days.",
    "Professional Development: ABC Corp provides an annual stipend of $1,500 per employee for professional development, including certifications, online courses, and conference attendance.",
    "Code of Conduct: Employees are expected to maintain a professional and inclusive workplace. Harassment or discrimination of any kind will not be tolerated and should be reported to the HR department immediately."
]

In [7]:
collection.upsert(
    documents=handbook_chunks,
    ids=[f'chunk_{i}' for i in range(len(handbook_chunks))]
)

print(f'Indexed {len(handbook_chunks)} chunks')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 86.3MiB/s]


Indexed 5 chunks


# RAG Query Function

In [16]:
def ask(question: str, top_k: int = 3):
  # Retrieve relevant chunks
  results = collection.query(
      query_texts=[question],
      n_results=top_k
  )

  received = results['documents'][0]
  context = '\n\n'.join(received)
  prompt = f"""
  You are a helpful assistant for ABC Corp. Use the provided context to answer the question at the end.
  If you do not know the answer based on the context, simply say that you do not know.
  Keep your answer concise and professional.

  Context:
  {context}

  Question:
  {question}

  Answer:
  """

  response = groq_client.chat.completions.create(
      model='llama-3.3-70b-versatile',
      messages=[{
          'role': 'user',
          'content': prompt
      }]
  )
  return response.choices[0].message.content

In [18]:
questions = [
    "What is the company's mission?",
    "How many days can I work remotely?",
    "What are the core business hours for remote employees?",
    "How much PTO do I get as a full-time employee?",
    "Do I need a doctor's note for sick leave?",
    "How much is the annual professional development stipend?",
    "What should I do if I witness workplace harassment?",
    "What are employee rights?",
    'Who is the founder of ABC'
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}\n{'-'*50}")

Q: What is the company's mission?
A: ABC Corp's mission is to innovate in the technology sector.
--------------------------------------------------
Q: How many days can I work remotely?
A: You can work remotely up to 3 days per week.
--------------------------------------------------
Q: What are the core business hours for remote employees?
A: The core business hours for remote employees are 10:00 AM to 3:00 PM EST.
--------------------------------------------------
Q: How much PTO do I get as a full-time employee?
A: As a full-time employee, you are entitled to 20 days of Paid Time Off (PTO) per year, accrued monthly.
--------------------------------------------------
Q: Do I need a doctor's note for sick leave?
A: You need a doctor's note for sick leave if your absence exceeds three consecutive days.
--------------------------------------------------
Q: How much is the annual professional development stipend?
A: The annual professional development stipend is $1,500 per employee.
----